# 03 -- Stage 3: Merge Image + Video Datasets

Combines `data/labels.csv` (Stage 1) and `data/dfdc_frame_manifest.csv`
(Stage 2) into a single manifest, then retrains on the union.

In [ ]:
import sys, os

def _is_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

IS_COLAB = _is_colab()
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import pandas as pd
from config.loader import resolve_path

image_labels = pd.read_csv(resolve_path("data/labels.csv"))
frame_manifest_path = resolve_path("data/dfdc_frame_manifest.csv")

if frame_manifest_path.exists():
    frame_labels = pd.read_csv(frame_manifest_path)
    merged = pd.concat([image_labels, frame_labels], ignore_index=True)
else:
    merged = image_labels

merged_path = resolve_path("data/merged_labels.csv")
merged.to_csv(merged_path, index=False)
print("Merged manifest:", merged_path, "--", len(merged), "rows")

Rerun the Stage 1 training loop pointed at `data/merged_labels.csv`,
loading `models/image_model.pth` as the starting checkpoint.